In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import csv
import pip
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import linearmodels as lm
from linearmodels import PanelOLS, RandomEffects
from scipy import stats
from linearmodels import RandomEffects
import statsmodels.api as sm
from linearmodels.panel import compare

In [ ]:
final_df = pd.read_csv("prepared_data_for_regression.csv")
print(final_df.head(5))

In [ ]:
initial_countries=final_df['country'].nunique()
print(initial_countries)

# OECD  Countries:

In [ ]:
oecd_countries = ['austria', 'australia', 'belgium', 'canada', 'chile', 
                  'colombia', 'czech', 'denmark', 'estonia', 'finland', 
                  'france', 'germany', 'greece', 'hungary', 'iceland', 'ireland', 
                  'israel', 'italy', 'japan', 'korea', 'latvia', 'lithuania', 'luxembourg', 
                  'mexico', 'netherlands', 'new zealand', 'norway', 'poland', 'portugal', 'slovakia', 
                  'slovenia', 'spain', 'sweden', 'switzerland', 'turkiye', 'united kingdom', 'united states', 'costa rica']
print(len(oecd_countries))

# Create Dummy Variables:

In [ ]:
final_df['is_oecd'] = final_df['country'].isin(oecd_countries).astype(int)

# create interaction :

In [ ]:
final_df['bmi_x_oecd'] = final_df['bmi']* final_df['is_oecd']
print(final_df['is_oecd'].value_counts())

# Dummy Variables: 

In [ ]:
final_df['is_oecd']= final_df['country'].str.lower().str.strip().isin(oecd_countries).astype(int)
print(final_df['is_oecd'].value_counts())

1. Aging Society Dummy: Based on WHO standards, societies with >10% population over 65 are considered "ageing" or "aged". This captures potential non-linear increases in cancer incidence due to population structure.

In [ ]:
final_df['dm_high_aging_society'] = (final_df['pop_65'] > 10).astype(int)
print(final_df['dm_high_aging_society'].value_counts())

2. High Life Expectancy Dummy threshold of 75 years represents advanced healthcare systems and higher probability of disease detection.

In [ ]:
print(final_df[['year', 'country', 'is_oecd', 'dm_high_aging_society']].head(100))

# interaction of is_oecd variable and gdp

In [ ]:
final_df['log_gdp_is_oecd']= final_df['log_gdp']*final_df['is_oecd']
print(final_df['log_gdp_is_oecd'].head(5))

In [ ]:
final_df['is_oecd'] = 0
final_df.loc[final_df['country'].isin(oecd_countries), 'is_oecd'] = 1

# interaction of is_oecd variable and gdp
final_df['log_gdp_is_oecd'] = final_df['log_gdp'] * final_df['is_oecd']
# Filling NaNs in Interaction Term: Since 'NaN * 0 = NaN' in Python, rows where is_oecd is 0 but log_gdp is missing(NAN) 
# would incorrectly stay as NaN (and be counted in the results), then filled these with 0 to ensure only real OECD countries with data are counted.
final_df['log_gdp_is_oecd'] = final_df['log_gdp_is_oecd'].fillna(0)

# counting contries :
oecd_count = final_df[final_df['is_oecd'] == 1]['country'].nunique()
interaction_countries = final_df[final_df['log_gdp_is_oecd'] != 0]['country'].nunique()

In [ ]:
print(final_df[['country','log_gdp','is_oecd','log_gdp_is_oecd','dm_high_aging_society']].sample(20).round(2))

# delete percentage of rows with missing value(NA):

In [ ]:
# report of missing values
missing_report = (final_df.isnull().sum() / len(final_df) * 100).round(2)
print(missing_report[missing_report > 0])

In [ ]:
clean_df= final_df.dropna(subset=['MIR', 'log_gdp', 'pop_65', 'urban_pop', 'fertility', 'labor_rate', 'log_gdp_is_oecd', 'internet', 'health_exp', 'smoking','hosp_beds'])
print(f" Missing values after deleting NA in clean_df: {clean_df.isnull().sum()}")

# Building stepwise regression

# Model 1: Adding Urban Population and log_gdp

In [ ]:
df_step= clean_df.set_index(['country', 'year'])
exog_1_fe= sm.add_constant(df_step[['log_gdp', 'urban_pop']])
model_1_fe=PanelOLS(df_step['MIR'],exog_1_fe, entity_effects=True, time_effects=True).fit()
print(model_1_fe.summary)

# Model 2: Adding Pop_65:

In [ ]:
exog_2_fe=sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65']])
model_2_fe = PanelOLS(df_step['MIR'], exog_2_fe, entity_effects=True, time_effects=True).fit()
print(model_2_fe.summary)

# Model 3: Adding Labor Rate:

In [ ]:
exog_3_fe=sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate']])
model_3_fe=PanelOLS(df_step['MIR'], exog_3_fe, entity_effects=True, time_effects=True).fit()
print(model_3_fe.summary)

# Model 4: Adding BMI:

In [ ]:
exog_4_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi']])
model_4_fe=PanelOLS(df_step['MIR'], exog_4_fe, entity_effects=True, time_effects= True).fit()
print(model_4_fe.summary)

# Model 5: Adding is_oecd as Dummy Variable:

In [ ]:
exog_5_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd']])
model_5_fe=PanelOLS(df_step['MIR'], exog_5_fe, entity_effects=True, time_effects=True).fit()
print(model_5_fe.summary)

# Model 6: Adding High aging society as Dummy Variable:

In [ ]:
exog_6_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society']])
model_6_fe = PanelOLS(df_step['MIR'], exog_6_fe, entity_effects=True, time_effects=True).fit()
print(model_6_fe.summary)

# create labor_rate * is_oecd variable:

In [ ]:
df_step['labor_rate * is_oecd']= df_step['labor_rate']* df_step['is_oecd']

# Model 7: Adding labor_rate * is_oecd:

In [ ]:
exog_7_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi',
                                     'log_gdp_is_oecd', 'dm_high_aging_society', 'labor_rate * is_oecd']])
model_7_fe = PanelOLS(df_step['MIR'], exog_7_fe, entity_effects=True, time_effects= True).fit()
print(model_7_fe.summary)

# Model 8: Adding internet:

In [ ]:
exog_8_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd',
                                     'dm_high_aging_society', 'labor_rate * is_oecd','internet']])
model_8_fe = PanelOLS(df_step['MIR'], exog_8_fe, entity_effects=True, time_effects= True).fit()
print(model_8_fe.summary)

# Model  9: Adding smoking:

In [ ]:
exog_9_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd',
                                     'dm_high_aging_society', 'labor_rate * is_oecd','internet','smoking']])
model_9_fe = PanelOLS(df_step['MIR'], exog_9_fe, entity_effects=True, time_effects= True).fit()
print(model_9_fe.summary)

# Model 10: adding health_exp:

In [ ]:
exog_10_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd','dm_high_aging_society',
                                      'labor_rate * is_oecd','internet','smoking','health_exp']])
model_10_fe = PanelOLS(df_step['MIR'], exog_10_fe, entity_effects=True, time_effects= True).fit()
print(model_10_fe.summary)

# Model 11: adding Fertality variable:

In [ ]:
exog_11_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 
                                      'bmi','log_gdp_is_oecd','dm_high_aging_society', 
                                      'labor_rate * is_oecd','internet','smoking','health_exp','fertility']])
model_11_fe = PanelOLS(df_step['MIR'], exog_11_fe, entity_effects=True, time_effects= True).fit()
print(model_11_fe.summary)

# Comparing results of all Fixed Effect models:

In [ ]:
Compare_results_fixed_Effect = compare({
    'FE model 1': model_1_fe,
    'FE model 2': model_2_fe,
    'FE model 3': model_3_fe,
    'FE model 4': model_4_fe,
    'FE model 5': model_5_fe,
    'FE model 6': model_6_fe,
    'FE model 7': model_7_fe,
    'FE model 8': model_8_fe,
    'FE model 9': model_9_fe,
    'FE model 10': model_10_fe,
    'FE model 11': model_11_fe})

print(Compare_results_fixed_Effect)

In [ ]:
with open('Comparison_MIR_Table_v1.html', 'w') as f:
    f.write(Compare_results_fixed_Effect.summary.as_html())